<a href="https://colab.research.google.com/github/karthik1338/workflowLLM_Demo/blob/main/demo_workflow_19_09_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WorkflowLLM refered from chatGPT
https://chatgpt.com/share/6aae991f-2724-83e9-93d9-7bbdefc50a74

##Phase 1 — Understand Dataset Structure


In [1]:
import json

with open("data_samples.json","r",encoding="utf-8") as f:
    data = json.load(f)

print("Samples:", len(data))

sample = data[0]

for k in sample.keys():
    print(k)

Samples: 50
query
apis
task_plan
annotated_code


In [2]:
# Samples: 50
# query
# apis
# task_plan
# annotated_code

##Phase 2 — Inspect Dataset Quality

We need to understand:

- average API count
- average plan size
- average code size

before choosing max sequence length.

In [3]:
for i in range(3):
    s = data[i]

    print("="*100)
    print("QUERY")
    print(s["query"][:300])

    print("\nAPI COUNT")
    print(len(s["apis"]))

    print("\nTASK PLAN LENGTH")
    print(len(s["task_plan"]))

    print("\nCODE LENGTH")
    print(len(s["annotated_code"]))

QUERY
What steps would I need to follow to develop a script that interacts with a Mastodon account? Specifically, I'm interested in how to extract a user's ID from their account link, check their current lists, and manage adding them to different lists.

API COUNT
17

TASK PLAN LENGTH
4614

CODE LENGTH
10039
QUERY
I'm interested in developing a simple clicker game where users can manage various features such as upgrading their clicks, activating boosts, redeeming codes for bonus clicks, and saving their progress into a text file. What considerations should I keep in mind for the design and functionality of su

API COUNT
17

TASK PLAN LENGTH
3205

CODE LENGTH
32684
QUERY
What steps can I follow to design a simulation that mimics a basic aerial attack scenario? I'm interested in aspects such as randomly generating coordinates for targeting, performing relevant calculations, overlaying images that represent aerial views, and managing alerts based on specific coordinat

API COUNT
11

TASK 

In [4]:
# ====================================================================================================
# QUERY
# What steps would I need to follow to develop a script that interacts with a Mastodon account? Specifically, I'm interested in how to extract a user's ID from their account link, check their current lists, and manage adding them to different lists.

# API COUNT
# 17

# TASK PLAN LENGTH
# 4614

# CODE LENGTH
# 10039
# ====================================================================================================
# QUERY
# I'm interested in developing a simple clicker game where users can manage various features such as upgrading their clicks, activating boosts, redeeming codes for bonus clicks, and saving their progress into a text file. What considerations should I keep in mind for the design and functionality of su

# API COUNT
# 17

# TASK PLAN LENGTH
# 3205

# CODE LENGTH
# 32684
# ====================================================================================================
# QUERY
# What steps can I follow to design a simulation that mimics a basic aerial attack scenario? I'm interested in aspects such as randomly generating coordinates for targeting, performing relevant calculations, overlaying images that represent aerial views, and managing alerts based on specific coordinat

# API COUNT
# 11

# TASK PLAN LENGTH
# 4392

# CODE LENGTH
# 7400


##Phase 3 — Compute Statistics

In [5]:
api_counts = []
plan_lengths = []
code_lengths = []

for s in data:
    api_counts.append(len(s["apis"]))
    plan_lengths.append(len(s["task_plan"]))
    code_lengths.append(len(s["annotated_code"]))

print("Avg APIs:", sum(api_counts)/len(api_counts))
print("Avg Plan Length:", sum(plan_lengths)/len(plan_lengths))
print("Avg Code Length:", sum(code_lengths)/len(code_lengths))

Avg APIs: 21.0
Avg Plan Length: 3184.72
Avg Code Length: 28730.52


In [6]:
# Avg APIs: 21.0
# Avg Plan Length: 3184.72
# Avg Code Length: 28730.52


##Phase 3 — Compute Statistics

``Query + APIs
      ↓
Task Plan + Workflow Code``

In [7]:
def build_example(item):

    apis = "\n".join(item["apis"])

    prompt = f"""
User Query:
{item['query']}

Available APIs:
{apis}

Generate:
1. Task Plan
2. Workflow Code
"""

    response = f"""
Task Plan:
{item['task_plan']}

Workflow Code:
{item['annotated_code']}
"""

    return {
        "prompt": prompt,
        "response": response
    }

##Phase 5 — Convert To Chat Format

For Qwen training:

In [8]:
def convert(item):

    apis = "\n".join(item["apis"])

    return {
        "messages": [
            {
                "role":"user",
                "content":
f"""User Query:
{item['query']}

Available APIs:
{apis}

Generate Task Plan and Workflow Code."""
            },
            {
                "role":"assistant",
                "content":
f"""Task Plan:
{item['task_plan']}

Workflow Code:
{item['annotated_code']}"""
            }
        ]
    }

##Phase 6 — Choose Model

For a Colab T4 (16 GB VRAM), use: `Qwen/Qwen2.5-3B-Instruct`

In [9]:
from transformers import AutoTokenizer
import json
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct"
)

with open("data_samples.json") as f:
    data = json.load(f)

lengths = []

for item in data:

    apis = "\n".join(item["apis"])

    text = f"""
User Query:
{item['query']}

Available APIs:
{apis}

Task Plan:
{item['task_plan']}

Workflow Code:
{item['annotated_code']}
"""

    tokens = tokenizer(text)["input_ids"]

    lengths.append(len(tokens))

print("Average:", np.mean(lengths))
print("Max:", max(lengths))
print("Min:", min(lengths))

Average: 7634.42
Max: 25354
Min: 1134


In [10]:
# Average: 7634.42
# Max: 25354
# Min: 1134

#What We Just Learned

Your dataset is actually very close to the WorkflowLLM format.

A typical example is:

`INPUT
------
Query
API List

OUTPUT
-------
Task Plan
Annotated Code`

which matches the paper's workflow representation:

w = {Q, D, P, A}

Q = Query
D = API Documentation
P = Task Plan
A = Annotated Code

Biggest Problem

Look at this:

Avg Code Length = 28,730 chars

A rough estimate:

1 token ≈ 4 chars

Therefore:

28730 / 4 ≈ 7180 tokens

Add:

Task Plan ≈ 800 tokens
Query ≈ 100 tokens
APIs ≈ 200 tokens

Total:

≈ 8k–10k tokens/sample

In [11]:
from transformers import AutoTokenizer
import json
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct"
)

with open("data_samples.json") as f:
    data = json.load(f)

plan_tokens = []
code_tokens = []

for item in data:
    plan_tokens.append(
        len(tokenizer(item["task_plan"]).input_ids)
    )

    code_tokens.append(
        len(tokenizer(item["annotated_code"]).input_ids)
    )

print("PLAN AVG:", np.mean(plan_tokens))
print("PLAN MAX:", max(plan_tokens))

print("CODE AVG:", np.mean(code_tokens))
print("CODE MAX:", max(code_tokens))

PLAN AVG: 748.56
PLAN MAX: 1107
CODE AVG: 6691.22
CODE MAX: 24452


In [12]:
# PLAN AVG: 748.56
# PLAN MAX: 1107
# CODE AVG: 6691.22
# CODE MAX: 24452


##Recommendation: Build WorkflowLLM in Stages

The paper trains on over 100k examples and much larger compute.

For a Colab T4 and 50 samples, I would not try to reproduce the full training pipeline immediately.

Instead:

##Stage 1 — Planner Model

Train:

Query + APIs
        ↓
Task Plan

Only.

Example:

Input:
User Query
API List

Output:
Task Plan

Your average task plan is only:

~3185 chars

which is manageable.

This gives you a working Workflow Planner very quickly.

In [13]:
import json

with open("data_samples.json") as f:
    data = json.load(f)

planner_data = []

for item in data:

    apis = "\n".join(item["apis"])

    planner_data.append({
        "messages": [
            {
                "role": "user",
                "content":
f"""User Query:
{item['query']}

Available APIs:
{apis}

Generate a task plan."""
            },
            {
                "role": "assistant",
                "content": item["task_plan"]
            }
        ]
    })

with open("planner_dataset.jsonl","w") as f:
    for row in planner_data:
        f.write(json.dumps(row) + "\n")

In [14]:
# r = 16
# lora_alpha = 32
# lora_dropout = 0.05

In [15]:
# max_seq_length = 2048

In [16]:
# Step 1: Install correct Unsloth and dependencies for Google Colab
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers triton bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-h3ta5a8t/unsloth_fe95b4f37a3b4286a51279629169a4d8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-h3ta5a8t/unsloth_fe95b4f37a3b4286a51279629169a4d8
  Resolved https://github.com/unslothai/unsloth.git to commit 768d644036a948441ba80b626c9e747c1fecfbae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


##Load Model

In [17]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1568: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

##Add LoRA

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.


##Convert Dataset

Your planner dataset should become:

In [19]:
import json

rows = []

with open("planner_dataset.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))

In [20]:
from datasets import Dataset

formatted = []

for row in rows:

    user_msg = row["messages"][0]["content"]
    assistant_msg = row["messages"][1]["content"]

    text = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ],
        tokenize=False,
    )

    formatted.append({"text": text})

dataset = Dataset.from_list(formatted)

##Training

In [21]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=20,
        learning_rate=2e-4,
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        output_dir="outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

In [22]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 20 | Total steps = 140
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.949674
2,1.953780
3,2.096035
4,1.844049
5,1.665881
6,1.653153
7,1.709923
8,1.604097
9,1.596424
10,1.620631


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-140/tokenizer_config.json.


TrainOutput(global_step=140, training_loss=0.5632448763852673, metrics={'train_runtime': 1431.4406, 'train_samples_per_second': 0.699, 'train_steps_per_second': 0.098, 'total_flos': 1.6547750797541376e+16, 'train_loss': 0.5632448763852673, 'epoch': 20.0})

##Save Adapter

In [23]:
model.save_pretrained("workflow_planner_lora")
tokenizer.save_pretrained("workflow_planner_lora")

Unsloth: Restored added_tokens_decoder metadata in workflow_planner_lora/tokenizer_config.json.


('workflow_planner_lora/tokenizer_config.json',
 'workflow_planner_lora/chat_template.jinja',
 'workflow_planner_lora/tokenizer.json')

In [ ]:
# Your loss curve:
# Start: ~1.95
# Middle: ~0.5
# End: ~0.02–0.04
# with
# Final train loss: 0.563
# looks like extreme overfitting.

# That's expected because:
# 50 examples
# 20 epochs
# 3B parameter model

# is a tiny dataset for a model this size.


In [24]:
# from datasets import Dataset

# dataset = dataset.shuffle(seed=42)

# train_dataset = dataset.select(range(40))
# val_dataset = dataset.select(range(40, 50))

In [ ]:
# Good News
# The training pipeline works
# workflow_planner_lora/
# WorkflowLLM Phase 1: -> completed
# Query + APIs → Task Plan

# Next Step: Run Inference

Load the adapter and test it.

In [25]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model.load_adapter("workflow_planner_lora")

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-1): 2 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=256, bias=True)
            (lora_dropout): Mod

In [27]:
prompt = '''
User Query:
Create a workflow that:
- Downloads a webpage
- Extracts all hyperlinks
- Filters PDF links
- Saves the PDF URLs to a text file

Available APIs:
downloadurl
getlinkfromhtml
filterfiles
savefile

Generate a task plan.'''

In [28]:
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.1,
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

User Query:
Create a workflow that:
- Downloads a webpage
- Extracts all hyperlinks
- Filters PDF links
- Saves the PDF URLs to a text file

Available APIs:
downloadurl
getlinkfromhtml
filterfiles
savefile

Generate a task plan.
assistant
1. **Start**
   - Begin the workflow.
2. **Download Initial Page**
   - Call the function `downloadurl` with the URL `http://www.example.com`
   - Store the result in variable `initial_page_download`
3. **Fetch Initial Content Links**
   - Call `getlinkfromhtml` with the `initial_page_download` result
   - Store the result in variable `content_links_fetched`
4. **Assign Content Links**
   - Assign `content_links_fetched` to `all_content_links`
5. **Fetch Final Links**
   - Call `getlinkfromhtml` with `all_content_links`
   - Store the result in `final_links_fetched`
6. **Filter Links (Extension Check)**
   - Call `filterfiles` with `final_links_fetched` based on extensio

In [29]:
def code_plan(prompt):
  messages = [
      {
          "role": "user",
          "content": prompt
      }
  ]

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
  )

  inputs = tokenizer(
      text,
      return_tensors="pt"
  ).to("cuda")

  outputs = model.generate(
      **inputs,
      max_new_tokens=1024,
      temperature=0.1,
  )

  print(
      tokenizer.decode(
          outputs[0],
          skip_special_tokens=True
      )
  )

In [30]:
prompt1 = """
User Query:
Create a workflow that:
- Checks an email inbox
- Downloads PDF attachments
- Stores them in a local folder
- Sends a summary notification

Available APIs:
checkemail
downloadattachment
savefile
notification

Generate a task plan.
"""

code_plan(prompt1)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

User Query:
Create a workflow that:
- Checks an email inbox
- Downloads PDF attachments
- Stores them in a local folder
- Sends a summary notification

Available APIs:
checkemail
downloadattachment
savefile
notification

Generate a task plan.

assistant
1. **Start**
   - Begin the workflow.
2. **Assign Variable**
   - Create the variable `folder_path` and assign it the value of the path to a folder named "New folders".
   - Equation: `folder_path = \\account\\New folders\\`
3. **Get Email Action**
   - Call the function `get_email_action` with the folder path and store the result in `email_action`.
   - Assign `email_action` to the variable `email_action`.
4. **Loop through Email Actions**
   - Enter a loop to process each item in `email_action`:
   - For each `item` in `email_action`:
     1. **Check if Item is a Folder**
        - If `is_folder_item` is true:
          - Get the folder name and assign i

In [36]:
prompt2= '''
User Query:
Create a workflow that:
- Reads a CSV file
- Extracts event names and dates
- Creates calendar entries
- Reports any invalid rows

Available APIs:
readfile
parsecsv
createevent
alert

Generate a task plan.
'''

code_plan(prompt2)


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

User Query:
Create a workflow that:
- Reads a CSV file
- Extracts event names and dates
- Creates calendar entries
- Reports any invalid rows

Available APIs:
readfile
parsecsv
createevent
alert

Generate a task plan.

assistant
1. **Start**
   - Begin the workflow.
2. **Define Variable**
   - Create ` eventCal ` as a dictionary mapping event names to their corresponding calendar identifiers.
3. **Assign Value**
   - Set ` x` to `eventCal`.
4. **Read File**
   - Call `readfile` with path `cal/event/tasks.txt` and store the result in `fileContent`.
5. **Split Content**
   - Use `splitfilecontent` on `fileContent` with separator `\n`, storing the result in `splitLines`.
6. **Assign Split Result**
   - Assign `splitLines` to `xLines`.
7. **Filter Out Empty Strings**
   - Use `filterrows` on `xLines` to remove empty strings, storing the result in `filteredLines`.
8. **Assign Filtered Result**
   - Set `csvDat